# Getting started

In [ ]:
# Install pyLEAFS first when this notebook runs somewhere it is not installed,
# such as Google Colab. Where pyLEAFS is already installed this does nothing.
try:
    import pyLEAFS
except ImportError:
    %pip install --quiet git+https://github.com/EternalTime/LEAFS.git

## Installation

```bash
git clone https://github.com/EternalTime/LEAFS.git
cd LEAFS
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install -e .
```

pyLEAFS requires Python 3.8+ (tested on 3.8 through 3.14); numpy and matplotlib are installed automatically.
Activate the virtual environment in every new terminal: it makes `pip` available and keeps the install out of a system Python that may refuse it.

## A first simulation

[`forager()`](#pyLEAFS.simulation.Simulation.forager) builds the v1 world - one resource field, one greedy-forager population - with the parameters of the `forager` applet:

In [ ]:
from pyLEAFS import Simulation

sim = Simulation.forager(seed=0)
sim.run(1000)
print(sim.populations[0].count, "agents alive")
print(sim.fields[0].total(), "resources")

`run` stops early if the population goes extinct; pass `stop_on_extinction=False` to advance a fixed number of steps regardless.

## Watching it live

[`Viewer`](#pyLEAFS.viewer.Viewer) opens an interactive matplotlib window:

```python
from pyLEAFS import Simulation, Viewer

Viewer(Simulation.forager(seed=0)).play()
```

Controls:

| | |
|---|---|
| spacebar | Pause / resume. |
| click empty space | Add a new agent at the cursor (works paused or running). |
| click on an agent | Select it: a ring appears, the side panel shows its state (fuel, age, harvested count, offspring, heading), and its recent trajectory is drawn as a trail. |

Clicking empty space while an agent is selected deselects it and adds an agent there.
The same controls drive a 3d world; see below for what a click means once the box can be rotated.

## The homogeneity knob

A single dimensionless parameter controls the environment.
The homogeneity $\Xi$ fixes the energy per resource, and with it how patchy or uniform the field is:

In [ ]:
sparse = Simulation.forager(seed=0, Xi=0.3)   # patchy: few, rich resources
dense  = Simulation.forager(seed=0, Xi=1.0)   # uniform: many, lean resources

print(sparse.fields[0].N_eq)   # equilibrium resources per region
print(dense.fields[0].N_eq)

Larger $\Xi$ means a more homogeneous world with more, lower-energy resources.
See [The model](theory.rst) for the definition and its role.

## Two or three dimensions

The core is dimension-agnostic, and the length of the grid `shape` selects the dimension:

```python
flat = Simulation.forager(seed=0, shape=(10, 10))      # 2d
solid = Simulation.forager(seed=0, shape=(10, 10, 10))  # 3d

solid.run(500)
```

[`Viewer`](#pyLEAFS.viewer.Viewer) follows suit: it reads the dimension from the grid and builds a flat or a rotatable box, with the same keys, the same side panel, and the same colours either way.

```python
from pyLEAFS import Simulation, Viewer

Viewer(Simulation.forager(seed=0, shape=(10, 10, 10))).play()
```

A 3d axes already spends left-drag on rotating the view, and a click on the screen is a line into the picture rather than a single point, so the mouse means this:

- Left-drag rotates the view, and rotating never selects or adds anything.
- A click that leaves the view where it was - press and release at effectively the same place - is a pick.
- A pick selects the agent nearest that line of sight, as long as it is within the same `select_radius` tolerance as in 2d, measured on screen.
- A pick that lands near no agent adds one on that line, at the depth of the centre of the world box.

A 3d world holds far more resources than a 2d one of the same width - the default `shape=(10, 10, 10)` carries about 125,000 of them - so the field is drawn as a translucent haze that fades with distance, letting you see the agents through the volume instead of only those on the near face.
That same abundance makes each frame slower than in 2d; a smaller `shape` or a smaller `Xi` keeps it brisk.

## Reproducibility

One `numpy.random.Generator`, threaded through from the seed, feeds every stochastic part of the simulation, so two runs with the same `seed` produce identical histories:

In [ ]:
a = Simulation.forager(seed=7); a.run(300, stop_on_extinction=False)
b = Simulation.forager(seed=7); b.run(300, stop_on_extinction=False)
assert a.populations[0].count == b.populations[0].count

## Tests

```bash
source .venv/bin/activate
pip install -e '.[test]'
pytest
```